In [ ]:
# Import Modules

import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
import librosa
import librosa.display
from IPython.display import Audio

import torchaudio
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    Wav2Vec2Model,
    Wav2Vec2Processor,
    Trainer,
    TrainingArguments,
    Wav2Vec2ForSequenceClassification
)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ejlok1/toronto-emotional-speech-set-tess")

print("Path to dataset files:", path)

In [ ]:
# Load the Dataset

paths = []
labels = []

for dirname, _, filenames in os.walk('/root/.cache/kagglehub/datasets/ejlok1/toronto-emotional-speech-set-tess/versions/1'):
    for filename in filenames:
        paths.append(os.path.join(dirname, filename))

        label = filename.split('_')[-1]
        label = label.split('.')[0]
        labels.append(label.lower())

    if len(paths) == 2800:
          print("hi")
          break

print('Dataset is Loaded')

In [ ]:
len(paths)

In [ ]:
labels[:5]

In [ ]:
# Create a dataframe
df = pd.DataFrame()
df['audio_paths'] = paths
df['labels'] = labels
df.head()

In [ ]:
df['labels'].value_counts()

In [ ]:
# Exploratory Data Analysis
sns.countplot(data=df, x='labels')

In [ ]:
def waveplot(data, sr, emotion):
    plt.figure(figsize=(10, 4))
    plt.title(emotion, size=20)
    librosa.display.waveshow(data, sr=sr)
    plt.show()

In [ ]:
def spectogram(data, sr, emotion):
    x = librosa.stft(data)
    xdb = librosa.amplitude_to_db(abs(x))

    plt.figure(figsize=(11, 4))
    plt.title(emotion, size=20)

    librosa.display.specshow(xdb, sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar()

In [ ]:
emotion = 'fear'

path = np.array(df['audio_paths'][df['labels'] == emotion])[0]

data, sampling_rate = librosa.load(path)

waveplot(data, sampling_rate, emotion)
spectogram(data, sampling_rate, emotion)

Audio(path)

In [ ]:
emotion = 'neutral'
path = np.array(df['audio_paths'][df['labels'] == emotion])[0]
data, sampling_rate = librosa.load(path)
waveplot(data, sampling_rate, emotion)
spectogram(data, sampling_rate, emotion)
Audio(path)

In [ ]:
emotion = 'happy'
path = np.array(df['audio_paths'][df['labels'] == emotion])[0]
data, sampling_rate = librosa.load(path)
waveplot(data, sampling_rate, emotion)
spectogram(data, sampling_rate, emotion)
Audio(path)

In [ ]:
# convert labels to integers
label_map = {label: idx for idx, label in enumerate(df['labels'].unique())}
inverse_label_map = {idx: label for label, idx in label_map.items()}

df['labels'] = df['labels'].map(label_map)

df.head(2)

In [ ]:
class SpeechEmotionDataset(Dataset):
    def __init__(self, df, processor, max_length=32000):
        self.df = df
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
      audio_path = self.df.iloc[idx]['audio_paths']
      label = self.df.iloc[idx]['labels']

      # load the audio file
      speech, sr = librosa.load(audio_path,sr=16000)


      # pad or truncate the speech to the required length
      if len(speech) > self.max_length:
          speech = speech[:self.max_length]
      else:
          speech = np.pad(speech, (0, self.max_length - len(speech)), 'constant')

      # preprocess the audio file
      inputs = self.processor(
          speech,
          sampling_rate=16000,
          return_tensors='pt',
          padding=True,
          truncate=True,
          max_length=self.max_length
      )

      input_values = inputs.input_values.squeeze()

      return {
          'input_values': input_values,
          'labels': torch.tensor(label, dtype=torch.long)
      }

In [ ]:

# split the data into train and test
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
train_dataset[0]['input_values'].size()

In [ ]:
# initialize the processor and model
processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base')
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    'facebook/wav2vec2-base',
    num_labels=7
)

In [ ]:
# load the dataset
train_dataset = SpeechEmotionDataset(train_df, processor)
test_dataset = SpeechEmotionDataset(test_df, processor)

In [ ]:
# create dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to=[]
)

In [ ]:
# create functions for computing metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids  # original labels
    preds = np.argmax(pred.predictions, axis=1)  # model predicted labels
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average='weighted'
    )
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
# initialize the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
import random

idx = random.randrange(0, len(test_dataset))
print("Original Label:", inverse_label_map[int(test_dataset[idx]['labels'])])

input_values = test_dataset[idx]['input_values'].unsqueeze(0).to('cuda')

with torch.no_grad():
    outputs = model(input_values)
    logits = outputs.logits

predicted_class = logits.argmax(dim=-1).item()
print("Predicted Label:", inverse_label_map[predicted_class])

In [ ]:
idx = random.randrange(0, len(test_dataset))
print("Original Label:", inverse_label_map[int(test_dataset[idx]['labels'])])

input_values = test_dataset[idx]['input_values'].unsqueeze(0).to('cuda')

with torch.no_grad():
    outputs = model(input_values)
    logits = outputs.logits

predicted_class = logits.argmax(dim=-1).item()
print("Predicted Label:", inverse_label_map[predicted_class])